<a href="https://colab.research.google.com/github/EmanElmaasarawi/Depi-Machine-learning/blob/main/copmaring_transformers_copy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util
import re
from collections import Counter

In [23]:
# ---------------- CONFIGURATION ----------------
SHEET1_PATH = "lookup_fr-veg.xlsx"  # Contains columns: name, code
SHEET2_PATH = "carrefour_Fruits_Veg_2026-03-15 09h56m.xlsx"  # Contains column: name
OUTPUT_PATH = "matched_output.xlsx"
SIMILARITY_THRESHOLD = 0.75  # Adjust based on your data quality
BATCH_SIZE = 64              # Reduce if you run out of memory
# ------------------------------------------------
# 1. Load data
df1 = pd.read_excel("lookup_fr-veg.xlsx")  # فيه name + code
df2 = pd.read_excel("carrefour_Fruits_Veg_2026-04-07 12h02m.xlsx")  # فيه name فقط


In [24]:
def clean_names(text):
    if pd.isna(text):
        return text


    # 1️⃣ Remove diacritics (tashkeel) + tatweel
    text = re.sub(r'[\u064B-\u065F\u0670\u0640]', '', text)

    # 2️⃣ Normalize Hamza forms → ا
    hamza_map = str.maketrans({'أ': 'ا', 'إ': 'ا', 'آ': 'ا'})
    text = text.translate(hamza_map)

    # 3️⃣ Normalize Alif Maqsura (ى) → ي
    text = text.replace('ى', 'ي')

    # 4️⃣ Normalize Ta Marbuta (ة) → ه
    text = text.replace('ة', 'ه')

    # 5️⃣ Remove punctuation except Arabic/English letters, numbers, spaces
    text = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', text, flags=re.UNICODE)

     # 6️⃣ Unit normalization (your dict + extensions)
    replacements = {
        # Weight
        r'\bكيلو\b': 'كجم',
        r'\bكيلوجرام\b': 'كجم',
        r'\b\(ك\)\b': 'كجم',
        r'\bkg\b': 'كجم',
        r'\bكلغ\b': 'كجم',
        r'\bجرام\b': 'جم',
        r'\bغرام\b': 'جم',
        r'\bغ\b': 'جم',
        r'\bgr\b': 'جم',
        r'\bg\b': 'جم',
        # Volume
        r'\bلتر\b': 'لتر',
        r'\b\(ل\)\b': 'لتر',
        r'\blt\b': 'لتر',
        r'\bl\b': 'لتر',
        r'\bml\b': 'مل',
        r'\bملي\b': 'مل',
        r'\bمللتر\b': 'مل',

        # Count
        r'\bحبه\b': 'حبه',
        r'\bقطعه\b': 'قطعه',
        r'\bقطعتين\b': '2 قطعه',

        # Pack
        r'\bعلبه?\b': 'علبه',
        r'\bكرتونه?\ب': 'كرتونه',
        # Remove standalone numbers (optional: keep if part of product name)

    }

    for k, v in replacements.items():
        text = text.replace(k, v, text, flags=re.IGNORECASE)

    text = re.sub(r"\s+", " ", text).strip()
    return text

def remove_parentheses_content(text):
    # الباترن: يبحث عن قوس فتح، أي محتوى، ثم قوس غلق
    pattern = r"\([^)]*\)"

    # الاستبدال بسلسلة فارغة
    cleaned_text = re.sub(pattern, "", text)

    # (اختياري) تنظيف المسافات الزائدة الناتجة عن الحذف
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

    return cleaned_text



In [25]:
  # افترض الأعمدة
#names1 = df1['name'].tolist()
#codes1 = df1['Form_Code'].tolist()
#names2 = df2['Product Name'].tolist()
# Validate required columns
assert "name" in df1.columns and "Form_Code" in df1.columns, "Sheet1 must have 'name' and 'code'"
assert "Product Name" in df2.columns, "Sheet2 must have 'name'"
# 2. Preprocess names
df1["name_clean"] = df1["name"].apply(remove_parentheses_content).apply(clean_names)
df2["name_clean"] = df2["Product Name"].apply(remove_parentheses_content).apply(clean_names)

In [26]:
print(df1["name_clean"])
print(df2["name_clean"])

0               الثوم قديم بلدي
1               الثوم جديد بلدي
2          بصل قديم متوسط الحجم
3          بصل جديد متوسط الحجم
4      بطاطس متوسطه الحجم تحمير
                ...            
61             زيتون اخضر تفاحي
62             زيتون اخضر عزيزي
63                    تفاح بلدي
64    تفاح مستورد احمر امريكاني
65      تفاح مستورد اصفر لبناني
Name: name_clean, Length: 66, dtype: object
0                  تفاح من رويال جالا
1                         بطاطس للقلي
2                            موز بلدي
3                                خيار
4                          طماطم بلدي
                    ...              
268      تمر مجدول بريميم جامبو 1 كجم
269    تمر المدينه روعه النخيل 400 جم
270             تمر مجدول جامبو 1 كجم
271              زادنا تمر سكري 1 كجم
272      Pico Nectarine 1 Kiلترoجمram
Name: name_clean, Length: 273, dtype: object


In [27]:
companies = ["زادنا","مافا","رويال","تبارك","روعه النخيل","لينا","لينه","عاليه","بيلكو","ابو عوف","مزارع","جالا"
             ,"لايف جرين","القصيم","نارمر","المدينه","المروه","الصفوه","الفرسان","المجد","النجمه","توب","بيكو","النخيل",
             "الربيع","المنار","الزهراء","المهيدب","المراعي","فارمز","عصير","عصائر","تمرتنا","تيمار","توب فاليو","بريميوم","جولدن"]

In [28]:
pattern = "|".join(companies)

df2 = df2[~df2["name_clean"].str.contains(pattern, na=False)]

In [29]:
#toknize the clean names using Regex
#pattern = r"(\d+\.?\d*)\s*(كجم|كلغ|جم|لتر|مل|kg|g|l|ml|'2 قطعه'|'قطعه ')"
pattern = r"(\d+\.?\d*)\s*(كجم|كلغ|جم|لتر|مل|kg|g|l|ml|قطعه|قطعه واحده|قطعه 2)"
df2[["Extracted_quantity","Extracted_unit"]] = df2["name_clean"].str.extract(pattern)

# 3. حذف الباترن من النص الأصلي (استبداله بسلسلة فارغة)
df2["name_clean"] = df2["name_clean"].str.replace(pattern, "", regex=True)

# 4. تنظيف المسافات الزائدة الناتجة عن الحذف
df2["name_clean"] = df2["name_clean"].str.replace(r"\s+", " ", regex=True).str.strip()
#df2["Extracted_quantity"] = pd.to_numeric(df2["Extracted_quantity"], errors='coerce')

if "size" not in df2.columns:
    df2[["Unit","Size"]] = df2["Unit"].str.extract(pattern)
df2["Size"]=df2["Size"].apply(clean_names)

discPattern = r"(\d+)%"
df2[["DiscountPercent"]] = df2["Discount"].str.extract(discPattern)

In [30]:
def tokenize_arabic(text):
    """تقسيم النص العربي لكلمات ذات معنى، مع إزالة الكلمات غير المهمة"""
    # كلمات توقف عربية شائعة (يمكن توسيعها)
    DESCRIPTIVE_WORDS = {
        'محلي', 'طازج', 'مجمد', 'طبيعي', 'عضوي',
        'مصري', 'سوري', 'لبناني', 'تركي', 'اسباني', 'هولندي',
        'مقشر', 'مغسول', 'جاهز', 'كامل', 'نصف', 'مقطع'
    }

    # كلمات ربط وأدوات (stop words)
    stop_words = {
        'و', 'في', 'من', 'الي', 'علي', 'ال', 'ل', 'ك', 'ب', 'او',
        'عن', 'مع', 'حتي', 'لكن', 'لان', 'اذا' }

    # دمج القائمتين
    all_stop = DESCRIPTIVE_WORDS | stop_words

    tokens = [t for t in text.split() if t not in stop_words and len(t) > 1]
    return tokens

In [31]:
pip install rapidfuzz

In [32]:
import torch
import numpy as np
import re
from rapidfuzz import fuzz, process

def tokenize_arabic(text):
    """Strips diacritics & returns clean Arabic/English tokens."""
    if not isinstance(text, str):
        return []
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)  # Remove tashkeel & tatweel
    return re.findall(r'[^\W_]+', text, re.UNICODE)

def compute_smart_similarity(cos_sim_matrix, names1, names2, fuzzy_threshold=0.80):
    """
    Hybrid matcher: BERT + Token Overlap + Fuzzy Typo Correction
    ✅ Fully PyTorch-native (accepts & returns tensors on the same device)
    """
    device = cos_sim_matrix.device
    n2, n1 = cos_sim_matrix.shape

    # Keep as PyTorch tensors from the start
    bert_scores = cos_sim_matrix.clone()
    final_scores = bert_scores.clone()

    # 1️⃣ Precompute tokens (Python sets are optimal for small name strings)
    tokens1 = [set(tokenize_arabic(n)) for n in names1]
    tokens2 = [set(tokenize_arabic(n)) for n in names2]

    # 2️⃣ Fuzzy matrix (CPU-optimized, then moved to GPU/device)
    fuzzy_np = process.cdist(names2, names1, scorer=fuzz.token_sort_ratio) / 100.0
    fuzzy_t = torch.from_numpy(fuzzy_np).to(device)

    # 3️⃣ Compute token overlap masks (NumPy -> PyTorch for speed)
    core_np = np.zeros((n2, n1), dtype=bool)
    overlap_np = np.zeros((n2, n1), dtype=np.float32)
    subset_np = np.zeros((n2, n1), dtype=bool)
    is_short_np = np.zeros((n2, n1), dtype=bool)

    for i, t2 in enumerate(tokens2):
        len_t2 = len(t2)
        for j, t1 in enumerate(tokens1):
            union = t1 | t2
            if not union: continue
            core_np[i, j] = len(t1 & t2) > 0
            overlap_np[i, j] = len(t1 & t2) / len(union)
            subset_np[i, j] = t1.issubset(t2) or t2.issubset(t1)
            is_short_np[i, j] = len(t1) <= 3 and len_t2 <= 3

    # Move masks to PyTorch device
    core_t = torch.from_numpy(core_np).to(device)
    overlap_t = torch.from_numpy(overlap_np).to(device)
    subset_t = torch.from_numpy(subset_np).to(device)
    is_short_t = torch.from_numpy(is_short_np).to(device)

    # 4️⃣ Apply your exact if/elif priority using native PyTorch tensor ops
    # Priority 1: Strong token match
    mask1 = core_t & ((overlap_t >= 0.6) | subset_t)
    final_scores[mask1] = torch.maximum(bert_scores[mask1], overlap_t[mask1] * 0.95) + 0.03

    # Priority 2: Typo correction (short names, no core overlap)
    mask2 = (~core_t) & is_short_t & (fuzzy_t > fuzzy_threshold)
    final_scores[mask2] = fuzzy_t[mask2]

    # Priority 3: No core overlap & not typo -> penalize
    mask3 = (~core_t) & (~mask1) & (~mask2)
    penalty3 = torch.where(bert_scores[mask3] > 0.75, 0.30, 0.20)
    final_scores[mask3] = torch.maximum(bert_scores[mask3] - penalty3, torch.tensor(0.0, device=device))

    # Priority 4: Weak overlap -> slight penalty
    mask4 = core_t & (~mask1) & (overlap_t < 0.4)
    final_scores[mask4] = torch.maximum(bert_scores[mask4] - 0.12, torch.tensor(0.0, device=device))

    return torch.clamp(final_scores, 0.0, 1.0)

In [33]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("fill-mask", model="CAMeL-Lab/bert-base-arabic-camelbert-ca")

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-ca
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.weight     | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias    | UNEXPECTED |  | 
cls.seq_relationship.weight  | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:

In [34]:
import torch
import gc
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # ✅ VERIFIED Arabic-optimized multilingual model
    model = SentenceTransformer("intfloat/multilingual-e5-base", device=device)

    BATCH_SIZE = 128

    # 🔑 E5 models perform best with "query: " or "passage: " prefixes.
    # For symmetric name matching, we can use "passage: " for both.
    names1 = [f"passage: {x}" for x in df1["name_clean"].tolist()]
    names2 = [f"passage: {x}" for x in df2["name_clean"].tolist()]

    print("🔹 Encoding Sheet1 names...")
    emb1 = model.encode(
        names1, batch_size=BATCH_SIZE, show_progress_bar=True,
        convert_to_tensor=True, normalize_embeddings=True
    )

    print("🔹 Encoding Sheet2 names...")
    emb2 = model.encode(
        names2, batch_size=BATCH_SIZE, show_progress_bar=True,
        convert_to_tensor=True, normalize_embeddings=True
    )

    # Free VRAM before large matrix ops
    torch.cuda.empty_cache()
    gc.collect()

    # 🔥 Cosine similarity via dot product (valid because embeddings are normalized)
    print("🔹 Computing similarity matrix...")
    if len(df1) * len(df2) > 50_000_000:
        print("⚠️ Large dataset (>50M pairs). Consider chunking or FAISS to avoid RAM OOM.")

    cos_sim = torch.mm(emb2, emb1.T)  # Shape: (len_df2, len_df1)

    # Apply your custom logic
    adj_sim = compute_smart_similarity(cos_sim, df1["name_clean"].values, df2["name_clean"].values)



    # Extract best matches
    max_scores, best_indices = adj_sim.max(dim=1)
    best_indices = best_indices.cpu().numpy()
    max_scores = max_scores.cpu().numpy()

    # Build results
    df2_result = df2.copy()
    df2_result["matched_name_sheet1"] = df1.iloc[best_indices]["name_clean"].values
    df2_result["code"] = df1.iloc[best_indices]["code"].values
    df2_result["similarity_score"] = np.round(max_scores, 4)

    df2_filtered = df2_result[df2_result["similarity_score"] >= SIMILARITY_THRESHOLD]
    df2_filtered = df2_filtered.sort_values("similarity_score", ascending=False)

    df2_filtered.to_excel(OUTPUT_PATH, index=False)
    print(f"\n✅ Matching complete. {len(df2_filtered)}/{len(df2)} names matched.")
    print(f"💾 Results saved to {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔹 Encoding Sheet1 names...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔹 Encoding Sheet2 names...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

🔹 Computing similarity matrix...

✅ Matching complete. 107/155 names matched.
💾 Results saved to matched_output.xlsx
